# Checking the dataset features



In [1]:
from datasets import load_dataset_builder
ds_builder = load_dataset_builder("glue", "sst2")
ds_builder1 = load_dataset_builder("glue", "mrpc")

# Inspect dataset description
print(ds_builder.info.splits)
print(ds_builder.info.features)
print(ds_builder1.info.features)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

{'train': SplitInfo(name='train', num_bytes=4681603, num_examples=67349, shard_lengths=None, dataset_name=None), 'validation': SplitInfo(name='validation', num_bytes=106252, num_examples=872, shard_lengths=None, dataset_name=None), 'test': SplitInfo(name='test', num_bytes=216640, num_examples=1821, shard_lengths=None, dataset_name=None)}
{'sentence': Value('string'), 'label': ClassLabel(names=['negative', 'positive']), 'idx': Value('int32')}
{'sentence1': Value('string'), 'sentence2': Value('string'), 'label': ClassLabel(names=['not_equivalent', 'equivalent']), 'idx': Value('int32')}


# Data preprocessing

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", "sst2") #try sst2 instead of mrpc
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)


def tokenize_function(example):
    return tokenizer(example["sentence"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# postprocess dataset

remove redundant columns, reformat

because after tokenizer we have:

`features: ['sentence', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask']`

only need `features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask']`

In [3]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels") #labels provided → automatically computes and returns the loss.
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

In [4]:
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=8, collate_fn=data_collator
)
eval_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=8, collate_fn=data_collator
)

#check
for batch in train_dataloader:
    break
print({k: v.shape for k, v in batch.items()})
len(train_dataloader)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'labels': torch.Size([8]), 'input_ids': torch.Size([8, 17]), 'token_type_ids': torch.Size([8, 17]), 'attention_mask': torch.Size([8, 17])}


8419

In [ ]:
#example output
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)
"""
tensor(0.9802, grad_fn=<NllLossBackward0>) torch.Size([8, 2])
we get 8 results with the logits for 2 labels each
"""

In [ ]:
from torch.optim import AdamW
#back propagation -> adjust weight with adam optimizer
optimizer = AdamW(model.parameters(), lr=5e-5)
"""
Backpropagation computes gradients
Adam uses those gradients to adjust weights
"""

In [6]:
from transformers import get_scheduler

num_epochs = 3
num_training_steps = num_epochs * len(train_dataloader) #total training step = #epochs * #batches
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

=============== The training loop ===============

extra note:
Modern Training Optimizations: To make your training loop even more efficient, consider:

Gradient Clipping: Add `torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)` before `optimizer.step()`

Mixed Precision: Use `torch.cuda.amp.autocast()` and `GradScaler` for faster training

Gradient Accumulation: Accumulate gradients over multiple batches to simulate larger batch sizes

Checkpointing: Save model checkpoints periodically to resume training if interrupted

In [7]:
#hardware check
import torch

device = torch.device("cuda")
model.to(device) #move model to GPU

#pretty progress bar
from tqdm.auto import tqdm

progress_bar = tqdm(range(num_training_steps))

#train
model.train() #enable train mode
for epoch in range(num_epochs):
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()} #move each batch to GPU
        outputs = model(**batch) #foward pass
        loss = outputs.loss
        loss.backward() #back propagation

        optimizer.step() #adjust weight
        lr_scheduler.step() #linearly decrease the lr
        optimizer.zero_grad() #reset the gradient
        progress_bar.update(1)

  0%|          | 0/25257 [00:00<?, ?it/s]

============= Evaluation loop ===============

In [ ]:
!pip install evaluate

In [ ]:
import evaluate
metric = evaluate.load("glue", "sst2")
model.eval()
for batch in eval_dataloader:
  batch = {k: v.to(device) for k, v in batch.items()}
  with torch.no_grad():
      outputs = model(**batch)
  logits = outputs.logits
  preds = torch.argmax(logits, dim=-1)
  metric.add_batch(predictions=preds, references=batch["labels"])
metric.compute()

Accelerate is something that enable us to train with multiple GPUs or TPUs

For now it is not possible T-T

# Saving the model and publish to HF

In [9]:
from huggingface_hub import notebook_login

notebook_login()

In [10]:
model.push_to_hub("bert-base-uncased-finetuned-sst2")
tokenizer.push_to_hub("bert-base-uncased-finetuned-sst2")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...8g06kpq/model.safetensors:   0%|          | 14.2kB /  438MB            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Sabiiii/bert-base-uncased-finetuned-sst2/commit/7c1d5a0baead6fc1f1fc9154bd0087993dee3e26', commit_message='Upload tokenizer', commit_description='', oid='7c1d5a0baead6fc1f1fc9154bd0087993dee3e26', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Sabiiii/bert-base-uncased-finetuned-sst2', endpoint='https://huggingface.co', repo_type='model', repo_id='Sabiiii/bert-base-uncased-finetuned-sst2'), pr_revision=None, pr_num=None)